In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!


## Docker Swarm

We want a 4-node swarm. Each node should have 8 cores, 8GB of memory. 

In [2]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 4
coresReq = 8
ramReq = 8

# we scale up the requirements a bit to account for the potential of others joining the selected site. 
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

['FIU', 'UTAH', 'STAR', 'CERN', 'RUTG', 'CLEM', 'KANS', 'MASS', 'MAX', 'TACC', 'SEAT', 'WASH', 'BRIST', 'EDUKY', 'EDC', 'AMST', 'PSC', 'MICH', 'GPN', 'HAWI', 'INDI', 'DALL', 'SRI', 'PRIN', 'GATECH', 'TOKY', 'ATLA', 'SALT', 'NCSA']


To avoid the scenario where all students joined the same site, the site selection is now random!

In [3]:
import random
siteName = random.choice(usableSite)
sliceName = "Swarmy"
print(siteName)

slice = fablib.new_slice(name=sliceName)
network_name = 'ramnet'

# Network

net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", 
                          site=siteName,
                          cores=coresReq,
                          ram=ramReq,
                          disk=30, 
                          image='default_ubuntu_22')
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)
    
slice.submit()   


Retry: 10, Time: 264 sec


ID,0b4e3110-0254-4a52-a4f3-f357083d2818
Name,Swarmy
Lease Expiration (UTC),2026-04-10 17:36:38 +0000
Lease Start (UTC),2026-04-09 17:36:38 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
2cd363c4-8f61-4f5d-a919-88cce2c0d802,node1,2,8,100,default_ubuntu_22,qcow2,mass-w2.fabric-testbed.net,MASS,ubuntu,2001:48e8:6401:3:f816:3eff:fed2:e0a5,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fed2:e0a5,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key
1ebacc98-a9c3-4d9b-bb80-819ef5df1c26,node2,2,8,100,default_ubuntu_22,qcow2,mass-w2.fabric-testbed.net,MASS,ubuntu,2001:48e8:6401:3:f816:3eff:fe0c:63c4,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fe0c:63c4,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key
5f8ba607-b4f0-4ee2-81f4-7d15605169bc,node3,2,8,100,default_ubuntu_22,qcow2,mass-w2.fabric-testbed.net,MASS,ubuntu,2001:48e8:6401:3:f816:3eff:fe7d:5b8a,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fe7d:5b8a,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key
b6ff09ca-6126-4ed5-b429-1144288342c8,node4,2,8,100,default_ubuntu_22,qcow2,mass-w2.fabric-testbed.net,MASS,ubuntu,2001:48e8:6401:3:f816:3eff:fe1b:9ef6,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fe1b:9ef6,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
1f02be6f-7167-4414-8511-f47ae555bf04,ramnet,L2,L2Bridge,MASS,None,192.168.1.0/24,Active,


ERROR:paramiko.transport:Exception (client): Error reading SSH protocol banner
ERROR:paramiko.transport:Traceback (most recent call last):
ERROR:paramiko.transport:  File "/home/fabric/.local/lib/python3.11/site-packages/paramiko/transport.py", line 2363, in _check_banner
ERROR:paramiko.transport:    buf = self.packetizer.readline(timeout)
ERROR:paramiko.transport:          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ERROR:paramiko.transport:  File "/home/fabric/.local/lib/python3.11/site-packages/paramiko/packet.py", line 395, in readline
ERROR:paramiko.transport:    buf += self._read_timeout(timeout)
ERROR:paramiko.transport:           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
ERROR:paramiko.transport:  File "/home/fabric/.local/lib/python3.11/site-packages/paramiko/packet.py", line 665, in _read_timeout
ERROR:paramiko.transport:    raise EOFError()
ERROR:paramiko.transport:EOFError
ERROR:paramiko.transport:
ERROR:paramiko.transport:During handling of the above exception, another exception occurred:
ERROR:p

Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
node1-nic-p1,p1,node1,ramnet,100,,06:1F:06:7D:4A:1A,enp7s0,enp7s0,config,None,4,HundredGigE0/0/0/9
node2-nic-p1,p1,node2,ramnet,100,,0E:9A:31:A3:8D:B4,enp7s0,enp7s0,config,None,4,HundredGigE0/0/0/9
node3-nic-p1,p1,node3,ramnet,100,,0E:07:0F:DC:B6:BF,enp7s0,enp7s0,config,None,4,HundredGigE0/0/0/9
node4-nic-p1,p1,node4,ramnet,100,,12:99:13:E2:E0:8C,enp7s0,enp7s0,config,None,4,HundredGigE0/0/0/9



Time to print interfaces 289 seconds


'0b4e3110-0254-4a52-a4f3-f357083d2818'

In [4]:
slice.wait_ssh()

True

In [8]:
from ipaddress import IPv4Network

for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)

    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24")
    )

**Keep rerunning the cell below until everyone can ping everyone!**

In [10]:
for i in range(1,nodesReq):
    src = slice.get_node(name="node" + str(i))
    for j in range(i + 1,nodesReq + 1):
        des = slice.get_node(name="node" + str(j))           
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f'ping -c 2 {des_addr}')  

node1 is pinging node2 at 192.168.1.2 ========
PING 192.168.1.2 (192.168.1.2) 56(84) bytes of data.
64 bytes from 192.168.1.2: icmp_seq=1 ttl=64 time=0.110 ms
64 bytes from 192.168.1.2: icmp_seq=2 ttl=64 time=0.077 ms

--- 192.168.1.2 ping statistics ---
2 packets transmitted, 2 received, 0% packet loss, time 1025ms
rtt min/avg/max/mdev = 0.077/0.093/0.110/0.016 ms
node1 is pinging node3 at 192.168.1.3 ========
PING 192.168.1.3 (192.168.1.3) 56(84) bytes of data.
64 bytes from 192.168.1.3: icmp_seq=1 ttl=64 time=0.262 ms
64 bytes from 192.168.1.3: icmp_seq=2 ttl=64 time=0.084 ms

--- 192.168.1.3 ping statistics ---
2 packets transmitted, 2 received, 0% packet loss, time 1015ms
rtt min/avg/max/mdev = 0.084/0.173/0.262/0.089 ms
node1 is pinging node4 at 192.168.1.4 ========
PING 192.168.1.4 (192.168.1.4) 56(84) bytes of data.
64 bytes from 192.168.1.4: icmp_seq=1 ttl=64 time=0.192 ms
64 bytes from 192.168.1.4: icmp_seq=2 ttl=64 time=0.058 ms

--- 192.168.1.4 ping statistics ---
2 packets

## Create Inventory File

We now create an inventory file based on our slice information. 
- One node should be placed into the `swarm_manager` group.
- The rest are placed into the `swarm_workers` group.

Programmatically design the generation of `inventory.yml` based on this information. 

In [14]:
from pathlib import Path

# Fixed node names and internal IPs for the FABRIC slice
node_defs = []

for node in slice.get_nodes():
    name = node.get_name()        
    ip_addr = node.get_interface(network_name=network_name).get_ip_addr()
    if name == "node1":
        node_defs.append({"name": name, "private_ip": ip_addr, "group": "swarm_manager"})
    else:
        node_defs.append({"name": name, "private_ip": ip_addr, "group": "swarm_workers"})
        
slice_key = fablib.get_default_slice_key()["slice_private_key_file"]
ssh_config = "/home/fabric/work/fabric_config/ssh_config"

# Collect node objects and python interpreters
for nd in node_defs:
    node = slice.get_node(nd["name"])
    stdout, stderr = node.execute(
        "python3 -c 'import sys; print(sys.executable)'",
        quiet=True
    )
    nd["node"] = node
    nd["python"] = stdout.strip()

# Build YAML inventory as text
lines = []
lines.append("all:")
lines.append("  vars:")
lines.append('    ansible_become: true')
lines.append(f'    ansible_ssh_private_key_file: "{slice_key}"')
lines.append(f'    ansible_ssh_common_args: "-F {ssh_config}"')
lines.append('    swarm_manager_ip: "192.168.1.1"')
lines.append('    swarm_registry: "192.168.1.1:5000"')
lines.append("")
lines.append("  children:")
lines.append("    swarm_manager:")
lines.append("      hosts:")

# Manager host(s)
for nd in node_defs:
    if nd["group"] == "swarm_manager":
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          internal_ip: "{nd["private_ip"]}"')

lines.append("")
lines.append("    swarm_workers:")
lines.append("      hosts:")

# Worker hosts
for nd in node_defs:
    if nd["group"] == "swarm_workers":
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          internal_ip: "{nd["private_ip"]}"')

inventory = "\n".join(lines) + "\n"

Path("playbook").mkdir(exist_ok=True)
Path("playbook/inventory.yml").write_text(inventory)

print("Wrote playbook/inventory.yml")
print(inventory)

[{'name': 'node1', 'private_ip': '192.168.1.1', 'group': 'swarm_manager'}, {'name': 'node2', 'private_ip': '192.168.1.2', 'group': 'swarm_workers'}, {'name': 'node3', 'private_ip': '192.168.1.3', 'group': 'swarm_workers'}, {'name': 'node4', 'private_ip': '192.168.1.4', 'group': 'swarm_workers'}]
Wrote playbook/inventory.yml
all:
  vars:
    ansible_become: true
    ansible_ssh_private_key_file: "/home/fabric/.ssh/slice_key"
    ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
    swarm_manager_ip: "192.168.1.1"
    swarm_registry: "192.168.1.1:5000"

  children:
    swarm_manager:
      hosts:
        node1:
          ansible_host: "2001:48e8:6401:3:f816:3eff:fed2:e0a5"
          ansible_user: "ubuntu"
          ansible_python_interpreter: "/usr/bin/python3"
          internal_ip: "192.168.1.1"

    swarm_workers:
      hosts:
        node2:
          ansible_host: "2001:48e8:6401:3:f816:3eff:fe0c:63c4"
          ansible_user: "ubuntu"
          ansible_python_i

 ## Playbook for Docker

 

In [25]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-docker.yml


PLAY [Install Docker Engine on all nodes] **************************************

TASK [Gathering Facts] *********************************************************
ok: [node3]
ok: [node4]
ok: [node2]
ok: [node1]

TASK [Install prerequisite packages] *******************************************
ok: [node4]
ok: [node3]
ok: [node2]
ok: [node1]

TASK [Create Docker keyring directory] *****************************************
ok: [node1]
ok: [node4]
ok: [node3]
ok: [node2]

TASK [Download Docker GPG key] *************************************************
ok: [node4]
ok: [node1]
ok: [node3]
ok: [node2]

TASK [Add Docker apt repository] ***********************************************
ok: [node4]
ok: [node1]
ok: [node3]
ok: [node2]

TASK [Install Docker Engine] ***************************************************
ok: [node4]
ok: [node3]
ok: [node2]
ok: [node1]

TASK [Ensure docker group exists] **********************************************
ok: [node4]
ok: [node3]
ok: [node1]
ok: [node2]

TASK [A

In [17]:
for node in slice.get_nodes():
    print(f"==== {node.get_name()} ====")
    stdout, stderr = node.execute("docker version", quiet=True);
    print(stdout)

==== node1 ====
Client: Docker Engine - Community
 Version:           29.4.0
 API version:       1.54
 Go version:        go1.26.1
 Git commit:        9d7ad9f
 Built:             Tue Apr  7 08:35:18 2026
 OS/Arch:           linux/amd64
 Context:           default

Server: Docker Engine - Community
 Engine:
  Version:          29.4.0
  API version:      1.54 (minimum version 1.40)
  Go version:       go1.26.1
  Git commit:       daa0cb7
  Built:            Tue Apr  7 08:35:18 2026
  OS/Arch:          linux/amd64
  Experimental:     false
 containerd:
  Version:          v2.2.2
  GitCommit:        301b2dac98f15c27117da5c8af12118a041a31d9
 runc:
  Version:          1.3.4
  GitCommit:        v1.3.4-0-gd6d73eb8
 docker-init:
  Version:          0.19.0
  GitCommit:        de40ad0

==== node2 ====
Client: Docker Engine - Community
 Version:           29.4.0
 API version:       1.54
 Go version:        go1.26.1
 Git commit:        9d7ad9f
 Built:             Tue Apr  7 08:35:18 2026
 OS/Arch: 

In [18]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-swarm.yml


PLAY [Initialize swarm on manager] *********************************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Check local swarm state] *************************************************
ok: [node1]

TASK [Initialize swarm] ********************************************************
changed: [node1]

TASK [Get worker join token] ***************************************************
ok: [node1]

TASK [Save swarm join data for later plays] ************************************
changed: [node1]

PLAY [Join worker nodes to swarm] **********************************************

TASK [Gathering Facts] *********************************************************
ok: [node4]
ok: [node2]
ok: [node3]

TASK [Check local swarm state] *************************************************
ok: [node4]
ok: [node2]
ok: [node3]

TASK [Join swarm as worker] ****************************************************
changed: [node3]
changed: [node2]
chan

In [19]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-registry.yml


PLAY [Deploy a Docker registry into the swarm] *********************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Create registry data directory on manager] *******************************
changed: [node1]

TASK [Check whether registry service already exists] ***************************
ok: [node1]

TASK [Create registry service] *************************************************
skipping: [node1]

TASK [Show registry service] ***************************************************
ok: [node1]

TASK [Print registry service list] *********************************************
ok: [node1] => {
    "registry_ls.stdout_lines": [
        "ID             NAME       MODE         REPLICAS   IMAGE          PORTS",
        "nzupmgbj4r4x   registry   replicated   1/1        registry:2     *:5000->5000/tcp",
        "fs4o4hddimzu   web        replicated   4/4        nginx:alpine   *:8080->80/tcp"
    ]
}

PLAY RECAP *******************

In [20]:
stdout, stderr = slice.get_node(name="node1").execute("curl 127.0.0.1:5000/v2/_catalog", quiet=True);
print(stdout)

{"repositories":[]}



In [27]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-ramcoin.yml


PLAY [Build and deploy ram_coin on Docker Swarm] *******************************

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Ensure git is installed] *************************************************
ok: [node1]

TASK [Clone or update ram_coin repository] *************************************
ok: [node1]

TASK [Build ram_coin images] ***************************************************
changed: [node1] => (item=hasher)
changed: [node1] => (item=rng)
changed: [node1] => (item=webui)
changed: [node1] => (item=worker)

TASK [Push ram_coin images] ****************************************************
changed: [node1] => (item=hasher)
changed: [node1] => (item=rng)
changed: [node1] => (item=webui)
changed: [node1] => (item=worker)

TASK [Show registry catalog] ***************************************************
ok: [node1]

TASK [Print registry catalog] **************************************************
ok: [node1] => {
    "registry_c

## Create SSH Tunnel

In [28]:
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [33]:
!cp /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz ~/ 
!cd; tar xzf fabric_ssh_tunnel_tools.tgz; cd fabric_ssh_tunnel_tools; chmod 600 slice_key fabric-bastion-key

In [32]:
!ls ~/fabric_ssh_tunnel_tools

fabric-bastion-key	slice_key      ssh_config
fabric-bastion-key.pub	slice_key.pub


In [35]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='8000'

# Username/node on FABRIC
target_host=f'{node.get_username()}@{node.get_management_ip()}'

print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

ssh  -L 0.0.0.0:5555:127.0.0.1:8000 -i slice_key -F ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fe1b:9ef6


In [36]:
slice.delete()